# Actividad 02 - Validacion temporal en Machine Learning financiero

Este notebook resume la actividad de validacion temporal aplicada a datos financieros. La logica principal fue trasladada a `src/validation_ts.py` para que el cuaderno quede limpio, reproducible y facil de presentar.

Objetivo: comparar validaciones tradicionales versus validaciones temporales, identificar leakage y evaluar modelos con walk-forward.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.validation_ts import (
    WalkForwardConfig,
    autocorrelation_summary,
    build_basic_next_return_dataset,
    build_time_series_features,
    compare_cv_strategies,
    compare_random_vs_temporal_split,
    configure_plots,
    label_leakage_example,
    load_market_returns,
    make_lagged_direction_dataset,
    nested_walk_forward_auc,
    plot_cv_comparison,
    plot_gap_scenarios,
    plot_nested_results,
    plot_split_strategies,
    plot_walk_forward,
    rolling_leakage_example,
    scaler_leakage_example,
    summarize_walk_forward,
    walk_forward_auc,
)

configure_plots()

## 1. Carga de datos

Usaremos retornos diarios de activos financieros. El ejemplo principal trabaja con SPY, pero la misma logica aplica a credit scoring cuando las observaciones tienen fecha de evaluacion y fecha de resultado.

In [ ]:
TICKERS = ["SPY", "AAPL", "JPM", "XOM", "GLD"]
prices, returns = load_market_returns(TICKERS, start="2010-01-01", end="2024-12-31")

print(f"Periodo: {prices.index[0].date()} - {prices.index[-1].date()}")
print(f"Observaciones: {len(prices):,} dias de trading")
print(f"Activos: {list(prices.columns)}")
prices.head()

## 2. Dependencia temporal

En finanzas, los datos no son independientes en el tiempo. Por eso no basta con mezclar filas al azar: hay que respetar el orden cronologico.

In [ ]:
autocorrelation_summary(returns["SPY"])

## 3. Split aleatorio versus split temporal

Un split aleatorio puede mezclar pasado y futuro. En problemas financieros esto puede sobreestimar el desempeno del modelo.

In [ ]:
X_basic, y_basic = build_basic_next_return_dataset(returns["SPY"])
compare_random_vs_temporal_split(X_basic, y_basic)

In [ ]:
plot_split_strategies();

## 4. Ejemplos de leakage

Leakage significa que el modelo recibe informacion que no estaria disponible al momento de tomar la decision. En finanzas esto es especialmente peligroso porque puede crear resultados artificialmente buenos.

In [ ]:
scaler_leakage_example(returns["SPY"])

In [ ]:
rolling_leakage_example(returns["SPY"])

In [ ]:
label_leakage_example(returns["SPY"])

## 5. Features temporales sin mirar el futuro

Las variables se construyen con rezagos y ventanas historicas desplazadas. El target representa el movimiento futuro.

In [ ]:
X, y = build_time_series_features(returns["SPY"])

print(f"Dataset: {X.shape[0]} observaciones, {X.shape[1]} features")
print(f"Periodo: {X.index[0].date()} - {X.index[-1].date()}")
print(f"Proporcion de dias alcistas: {y.mean():.2%}")
X.head()

## 6. Comparacion de estrategias de validacion

K-Fold es util en muchos contextos, pero TimeSeriesSplit respeta mejor la logica de entrenamiento con pasado y evaluacion con futuro.

In [ ]:
cv_scores = compare_cv_strategies(X, y, n_splits=10)
cv_scores.groupby("estrategia")["auc"].agg(["mean", "std", "min", "max"])

In [ ]:
plot_cv_comparison(cv_scores);

## 7. Gap temporal

El gap deja una distancia entre train y test. Es util cuando existen variables rolling, rezagos o informacion que podria solaparse entre ventanas.

In [ ]:
plot_gap_scenarios();

## 8. Walk-forward validation

El modelo se entrena con datos pasados, se prueba en el siguiente periodo y luego la ventana avanza. Esto simula mejor el uso real de un modelo financiero.

In [ ]:
X_wf, y_wf = make_lagged_direction_dataset(returns["SPY"])
config = WalkForwardConfig(initial_train=504, test_size=63, step=21, gap=5)

wf_results = walk_forward_auc(X_wf, y_wf, config)
wf_results.head()

In [ ]:
summarize_walk_forward(wf_results)

In [ ]:
plot_walk_forward(wf_results);

## 9. Nested walk-forward

Nested walk-forward separa dos tareas: tuning interno de hiperparametros y evaluacion externa out-of-time. Esto reduce el riesgo de elegir parametros mirando indirectamente el test.

In [ ]:
nested_results = nested_walk_forward_auc(X_wf, y_wf, config)
nested_results.head()

In [ ]:
nested_results[["inner_auc", "outer_auc", "brecha_inner_outer"]].agg(["mean", "std"])

In [ ]:
plot_nested_results(nested_results);

## 10. Conclusion

Para proyectos financieros, el punto clave es evaluar como si el modelo estuviera operando en la realidad: aprende del pasado y se prueba en el futuro. Esto es directamente aplicable al proyecto de credit scoring, donde las variables deben estar disponibles antes de observar la mora o default.